# Model Training and Evaluation

This notebook handles model training, hyperparameter tuning, and evaluation for the sentiment analysis task.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import joblib
import os

plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'data_loader'

## 1. Load Prepared Data

In [ ]:
# Load data
try:
    train_df = pd.read_csv('../data/train.csv')
    val_df = pd.read_csv('../data/val.csv')
    test_df = pd.read_csv('../data/test.csv')
    print("Data splits loaded successfully!")
except FileNotFoundError:
    print("Data splits not found. Loading and preparing data...")
    df = pd.read_csv('../data/IMDB Dataset.csv')
    df = df.drop_duplicates().dropna()
    
    from sklearn.model_selection import train_test_split
    train_val_df, test_df = train_test_split(df, test_size=0.2, stratify=df['sentiment'], random_state=42)
    val_size_adj = 0.1 / (1 - 0.2)
    train_df, val_df = train_test_split(train_val_df, test_size=val_size_adj, stratify=train_val_df['sentiment'], random_state=42)
    
    # Save splits
    os.makedirs('../data', exist_ok=True)
    train_df.to_csv('../data/train.csv', index=False)
    val_df.to_csv('../data/val.csv', index=False)
    test_df.to_csv('../data/test.csv', index=False)

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

# Display sentiment distribution
print("\nSentiment distribution:")
for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    sentiment_dist = split_df['sentiment'].value_counts()
    print(f"{split_name}: {sentiment_dist.to_dict()}")

## 2. Feature Engineering

In [ ]:
# Feature engineering setup
max_features = 10000
ngram_range = (1, 2)
min_df = 5
max_df = 0.9

print("Feature engineering parameters:")
print(f"  Max features: {max_features}")
print(f"  N-gram range: {ngram_range}")
print(f"  Min document frequency: {min_df}")
print(f"  Max document frequency: {max_df}")

In [ ]:
# Text preprocessing function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing
print("Processing training data...")
start_time = time.time()

X_train_text = train_df['review'].apply(clean_text)
X_val_text = val_df['review'].apply(clean_text)
X_test_text = test_df['review'].apply(clean_text)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=max_features,
    ngram_range=ngram_range,
    min_df=min_df,
    max_df=max_df,
    stop_words='english'
)

# Fit and transform
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

# Encode labels
label_map = {'positive': 1, 'negative': 0}
y_train = train_df['sentiment'].map(label_map)
y_val = val_df['sentiment'].map(label_map)
y_test = test_df['sentiment'].map(label_map)

train_time = time.time() - start_time
print(f"Training data processed in {train_time:.2f} seconds")
print(f"Training features shape: {X_train.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

print(f"Validation features shape: {X_val.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Display feature engineering information
print("Feature Engineering Information:")
print(f"  Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"  Max features: {max_features}")
print(f"  Actual features: {X_train.shape[1]}")

# Display sample feature names
feature_names = vectorizer.get_feature_names_out()
print(f"\nSample feature names (first 20):")
for i, feature in enumerate(feature_names[:20]):
    print(f"  {i+1:2d}. {feature}")

## 3. Model Training

In [ ]:
# Model configurations
models = {
    'logreg': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {'C': [0.1, 1.0, 10.0]}
    },
    'svm': {
        'model': LinearSVC(random_state=42, max_iter=2000),
        'params': {'C': [0.1, 1.0, 10.0]}
    },
    'rf': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [10, 20, None]
        }
    }
}

print("Available models:")
for model_name in models.keys():
    print(f"  - {model_name}")

print("\nHyperparameter grids:")
for model_name, config in models.items():
    print(f"\n{model_name}:")
    for param, values in config['params'].items():
        print(f"  {param}: {values}")

In [ ]:
# Train all models
print("Starting model training...")
print("This may take some time depending on your system performance.")

results = {}
total_training_time = 0

for name, config in models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    
    # Setup cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Grid search
    grid = GridSearchCV(
        config['model'], 
        config['params'], 
        cv=cv, 
        scoring='f1', 
        n_jobs=-1
    )
    
    # Fit model
    grid.fit(X_train, y_train)
    
    # Get best model
    best_model = grid.best_estimator_
    
    # Validation predictions
    val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, val_pred)
    val_acc = accuracy_score(y_val, val_pred)
    
    # Training predictions
    train_pred = best_model.predict(X_train)
    train_f1 = f1_score(y_train, train_pred)
    train_acc = accuracy_score(y_train, train_pred)
    
    # Cross-validation scores
    cv_scores = grid.cv_score_ if hasattr(grid, 'cv_score_') else [grid.best_score_] * 5
    cv_mean = grid.best_score_
    cv_std = np.std(cv_scores)
    
    training_time = time.time() - start_time
    total_training_time += training_time
    
    results[name] = {
        'model': best_model,
        'best_params': grid.best_params_,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'cv_scores': cv_scores,
        'val_metrics': {
            'f1_score': val_f1,
            'accuracy': val_acc
        },
        'train_metrics': {
            'f1_score': train_f1,
            'accuracy': train_acc
        },
        'training_time': training_time
    }
    
    print(f"  Training time: {training_time:.2f}s")
    print(f"  CV F1-score: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"  Validation F1-score: {val_f1:.4f}")
    print(f"  Validation Accuracy: {val_acc:.4f}")

# Find best model
best_model_name = max(results.keys(), key=lambda x: results[x]['val_metrics']['f1_score'])
best_score = results[best_model_name]['val_metrics']['f1_score']

print(f"\nTraining completed in {total_training_time:.2f} seconds")
print(f"Best model: {best_model_name}")
print(f"Best validation F1-score: {best_score:.4f}")

In [ ]:
# Create comparison dataframe
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'model': name,
        'val_f1': result['val_metrics']['f1_score'],
        'val_accuracy': result['val_metrics']['accuracy'],
        'training_time': result['training_time'],
        'cv_mean_f1': result['cv_mean']
    })

comparison_df = pd.DataFrame(comparison_data)
print("Model Comparison:")
print(comparison_df.round(4))

# Visualize model comparison
plt.figure(figsize=(15, 10))

# F1-score comparison
plt.subplot(2, 2, 1)
sns.barplot(data=comparison_df, x='model', y='val_f1')
plt.title('Validation F1-Score Comparison')
plt.xlabel('Model')
plt.ylabel('F1-Score')
plt.xticks(rotation=45)

# Accuracy comparison
plt.subplot(2, 2, 2)
sns.barplot(data=comparison_df, x='model', y='val_accuracy')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)

# Training time comparison
plt.subplot(2, 2, 3)
sns.barplot(data=comparison_df, x='model', y='training_time')
plt.title('Training Time Comparison')
plt.xlabel('Model')
plt.ylabel('Training Time (seconds)')
plt.xticks(rotation=45)

# Cross-validation score comparison
plt.subplot(2, 2, 4)
sns.barplot(data=comparison_df, x='model', y='cv_mean_f1')
plt.title('Cross-Validation F1-Score Comparison')
plt.xlabel('Model')
plt.ylabel('CV F1-Score')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 4. Best Model Analysis

In [ ]:
# Analyze the best model
best_results = results[best_model_name]

print(f"=== BEST MODEL ANALYSIS: {best_model_name.upper()} ===")
print(f"\nBest hyperparameters:")
for param, value in best_results['best_params'].items():
    print(f"  {param}: {value}")

print(f"\nCross-validation scores:")
print(f"  Mean F1-score: {best_results['cv_mean']:.4f}")
print(f"  Std F1-score: {best_results['cv_std']:.4f}")
print(f"  All CV scores: {[f'{score:.4f}' for score in best_results['cv_scores']]}")

print(f"\nValidation metrics:")
for metric, value in best_results['val_metrics'].items():
    print(f"  {metric}: {value:.4f}")

print(f"\nTraining metrics:")
for metric, value in best_results['train_metrics'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Feature importance analysis (if available)
best_model = best_results['model']

if hasattr(best_model, 'feature_importances_'):
    # Get feature importance
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("Top 20 Most Important Features:")
    print(feature_importance_df.head(20))
    
    # Visualize top features
    plt.figure(figsize=(12, 8))
    top_features = feature_importance_df.head(20)
    
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.title(f'Top 20 Most Important Features - {best_model_name}')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type")

## 5. Model Evaluation on Test Set

In [ ]:
# Evaluate the best model on test set
print(f"Evaluating {best_model_name} on test set...")

# Make predictions
test_pred = best_model.predict(X_test)
test_f1 = f1_score(y_test, test_pred)
test_acc = accuracy_score(y_test, test_pred)

# Get probabilities if available
test_prob = None
if hasattr(best_model, 'predict_proba'):
    test_prob = best_model.predict_proba(X_test)[:, 1]
elif hasattr(best_model, 'decision_function'):
    decision_scores = best_model.decision_function(X_test)
    # Normalize to [0,1] range
    test_prob = (decision_scores - decision_scores.min()) / (decision_scores.max() - decision_scores.min())

# Calculate ROC-AUC if probabilities available
test_roc_auc = None
if test_prob is not None:
    test_roc_auc = roc_auc_score(y_test, test_prob)

test_results = {
    'metrics': {
        'f1_score': test_f1,
        'accuracy': test_acc,
        'roc_auc': test_roc_auc if test_roc_auc is not None else 'N/A'
    },
    'predictions': test_pred,
    'probabilities': test_prob if test_prob is not None else np.zeros(len(test_pred)),
    'true_labels': y_test,
    'class_names': ['negative', 'positive']
}

print(f"\nTest Results for {best_model_name}:")
for metric, value in test_results['metrics'].items():
    print(f"  {metric}: {value}")

In [ ]:
# Display detailed classification report
print("Detailed Classification Report:")
print(classification_report(
    test_results['true_labels'],
    test_results['predictions'],
    target_names=test_results['class_names']
))

# Display confusion matrix
print("Confusion Matrix:")
cm = confusion_matrix(test_results['true_labels'], test_results['predictions'])
print(cm)

if len(test_results['class_names']) == 2:
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix Details:")
    print(f"  True Negatives: {tn}")
    print(f"  False Positives: {fp}")
    print(f"  False Negatives: {fn}")
    print(f"  True Positives: {tp}")
    
    # Calculate additional metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    print(f"\nAdditional Metrics:")
    print(f"  Precision (Positive): {precision:.4f}")
    print(f"  Recall (Positive): {recall:.4f}")
    print(f"  Specificity (Negative): {specificity:.4f}")

## 6. Error Analysis

In [ ]:
# Analyze prediction errors
test_df_analysis = test_df.copy()
test_df_analysis['predicted_sentiment'] = test_df_analysis['sentiment'].map({0: 'negative', 1: 'negative'}).iloc[:len(test_pred)]
test_df_analysis['predicted_sentiment'] = ['positive' if p == 1 else 'negative' for p in test_pred]
test_df_analysis['prediction_probability'] = test_results['probabilities']
test_df_analysis['true_label_encoded'] = test_results['true_labels']

# Identify misclassified examples
misclassified = test_df_analysis[
    test_df_analysis['sentiment'] != test_df_analysis['predicted_sentiment']
]

print(f"Total test samples: {len(test_df_analysis)}")
print(f"Correctly classified: {len(test_df_analysis) - len(misclassified)}")
print(f"Misclassified: {len(misclassified)}")
print(f"Error rate: {len(misclassified)/len(test_df_analysis)*100:.2f}%")

# Display some misclassified examples
print(f"\n=== MISCLASSIFIED EXAMPLES ===")
for i, (_, row) in enumerate(misclassified.head(5).iterrows()):
    print(f"\nExample {i+1}:")
    print(f"True sentiment: {row['sentiment']}")
    print(f"Predicted sentiment: {row['predicted_sentiment']}")
    print(f"Prediction probability: {row['prediction_probability']:.4f}")
    print(f"Review: {row['review'][:200]}...")
    print("-" * 80)

In [ ]:
# Analyze confidence distribution
plt.figure(figsize=(15, 5))

# Overall confidence distribution
plt.subplot(1, 3, 1)
plt.hist(test_results['probabilities'], bins=50, alpha=0.7, color='skyblue')
plt.title('Prediction Confidence Distribution')
plt.xlabel('Prediction Probability')
plt.ylabel('Frequency')

# Confidence for correct predictions
correct_mask = test_df_analysis['sentiment'] == test_df_analysis['predicted_sentiment']
plt.subplot(1, 3, 2)
plt.hist(test_df_analysis[correct_mask]['prediction_probability'], 
         bins=50, alpha=0.7, color='green', label='Correct')
plt.hist(test_df_analysis[~correct_mask]['prediction_probability'], 
         bins=50, alpha=0.7, color='red', label='Incorrect')
plt.title('Confidence by Prediction Accuracy')
plt.xlabel('Prediction Probability')
plt.ylabel('Frequency')
plt.legend()

# Confidence by true sentiment
plt.subplot(1, 3, 3)
for sentiment in test_df_analysis['sentiment'].unique():
    sentiment_data = test_df_analysis[test_df_analysis['sentiment'] == sentiment]
    plt.hist(sentiment_data['prediction_probability'], 
             bins=50, alpha=0.7, label=sentiment)
plt.title('Confidence by True Sentiment')
plt.xlabel('Prediction Probability')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

# Confidence statistics
print("Confidence Statistics:")
print(f"Overall mean confidence: {np.mean(test_results['probabilities']):.4f}")
print(f"Correct predictions mean confidence: {np.mean(test_df_analysis[correct_mask]['prediction_probability']):.4f}")
print(f"Incorrect predictions mean confidence: {np.mean(test_df_analysis[~correct_mask]['prediction_probability']):.4f}")

## 7. Save Model and Results

In [ ]:
# Save the best model
os.makedirs('../models', exist_ok=True)
model_save_path = '../models/best_model.joblib'

model_data = {
    'model': best_model,
    'vectorizer': vectorizer,
    'model_name': best_model_name,
    'label_map': label_map,
    'metrics': test_results['metrics']
}

joblib.dump(model_data, model_save_path)
print(f"Best model saved to {model_save_path}")

# Save evaluation results
os.makedirs('../evaluation_results', exist_ok=True)
results_path = '../evaluation_results/test_results.joblib'
joblib.dump(test_results, results_path)
print(f"Evaluation results saved to {results_path}")

print("\n=== FINAL EVALUATION REPORT ===")
print(f"Best Model: {best_model_name}")
print(f"Test Accuracy: {test_results['metrics']['accuracy']:.4f}")
print(f"Test F1-Score: {test_results['metrics']['f1_score']:.4f}")
if test_results['metrics']['roc_auc'] != 'N/A':
    print(f"Test ROC-AUC: {test_results['metrics']['roc_auc']:.4f}")

## 8. Model Performance Summary

In [ ]:
# Create a comprehensive performance summary
performance_summary = {
    'best_model': {
        'name': best_model_name,
        'hyperparameters': best_results['best_params'],
        'validation_f1': best_results['val_metrics']['f1_score'],
        'test_f1': test_results['metrics']['f1_score'],
        'test_accuracy': test_results['metrics']['accuracy'],
        'test_roc_auc': test_results['metrics'].get('roc_auc', 'N/A')
    },
    'feature_engineering': {
        'vocabulary_size': len(vectorizer.vocabulary_),
        'max_features': max_features,
        'ngram_range': ngram_range
    },
    'training_info': {
        'total_training_time': total_training_time,
        'cross_validation_f1_mean': best_results['cv_mean'],
        'cross_validation_f1_std': best_results['cv_std']
    },
    'data_info': {
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'feature_dimensions': X_train.shape[1]
    }
}

print("=== MODEL PERFORMANCE SUMMARY ===")
print(f"\nBest Model: {performance_summary['best_model']['name']}")
print(f"Test Accuracy: {performance_summary['best_model']['test_accuracy']:.4f}")
print(f"Test F1-Score: {performance_summary['best_model']['test_f1']:.4f}")
if performance_summary['best_model']['test_roc_auc'] != 'N/A':
    print(f"Test ROC-AUC: {performance_summary['best_model']['test_roc_auc']:.4f}")

print(f"\nFeature Engineering:")
print(f"  Vocabulary Size: {performance_summary['feature_engineering']['vocabulary_size']}")
print(f"  Feature Dimensions: {performance_summary['data_info']['feature_dimensions']}")

print(f"\nTraining:")
print(f"  Total Time: {performance_summary['training_info']['total_training_time']:.2f} seconds")
print(f"  CV F1-Score: {performance_summary['training_info']['cross_validation_f1_mean']:.4f} ± {performance_summary['training_info']['cross_validation_f1_std']:.4f}")

print(f"\nData:")
print(f"  Train/Val/Test samples: {performance_summary['data_info']['train_samples']}/{performance_summary['data_info']['val_samples']}/{performance_summary['data_info']['test_samples']}")

print("\n=== CONCLUSION ===")
if performance_summary['best_model']['test_f1'] > 0.85:
    print("✓ Excellent model performance achieved!")
elif performance_summary['best_model']['test_f1'] > 0.80:
    print("✓ Good model performance achieved.")
elif performance_summary['best_model']['test_f1'] > 0.75:
    print("✓ Acceptable model performance.")
else:
    print("⚠ Model performance could be improved.")

print("\nModel is ready for deployment!")